# Load imports

In [ ]:
import skillcorner as sc
import pandas as pd
import numpy as np
import mplsoccer
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
from pathlib import Path
import json
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
from requests import HTTPError
from matplotlib.animation import FuncAnimation
from skillcorner.client import SkillcornerClient



# SkillCorner Client init

Here I will view all the competitions available to me via the api and filter for the final ones I want to inlcude in my analysis. 

In [13]:
# Load credentials from JSON file in creds/
creds_path = Path("creds") / "skillcorner.json"
with creds_path.open() as f:
    sc_creds = json.load(f)

username = sc_creds["username"]
password = sc_creds["pswd"]

client = SkillcornerClient(username=username, password=password)

In [ ]:
# Fetch all competitions you have access to
competitions = client.get_competitions()

# Build a set of unique competition names (fallback for key name)
# and keep only North American competitions (e.g. MLS, USL, NWSL, etc.)
na_keywords = [
    "Major League Soccer",
    "MLS",
    "USL",
    "NWSL",
    "Canadian Premier League",
    "Liga MX",
    "Liga MX Femenil",
    "Leagues Cup",
    "Gold Cup",
    "Road to W Gold Cup",
    "Women's Gold Cup",
    "Liga de Expansión MX",
    "Liga de Ascenso",
]

def is_north_america(name: str) -> bool:
    lname = name.lower()
    return any(k.lower() in lname for k in na_keywords)

unique_competitions = sorted({
    name
    for c in competitions
    for name in [c.get("name") or c.get("competition_name")]
    if name and is_north_america(name)
})

print(f"Found {len(unique_competitions)} unique North American competitions:")
for comp in unique_competitions:
    print(comp)

Found 21 unique North American competitions:
Canadian Premier League
Gold Cup
Gold Cup Qualification
Leagues Cup
Liga MX
Liga MX Femenil
Liga de Ascenso
Liga de Expansión MX
MLS Next Pro
MLSZ U15 Országos Kiemelt
MLSZ U16 Országos Kiemelt
MLSZ U19 Országos Kiemelt
Major League Soccer
NWSL
NWSL Challenge Cup
Road to W Gold Cup - Play-Offs
USL Championship
USL League 1
USL Super League, Women
Veikkausliiga
Women's Gold Cup


FIlter for competitions I want to keep.

In [33]:
final_competitions = {
    "USL Championship"
}
# Find their competition IDs from the full competitions list
selected = [
    (c.get("name") or c.get("competition_name"), c["id"])
    for c in competitions
    if (c.get("name") or c.get("competition_name")) in final_competitions
]

In [35]:
print("Selected competitions and their IDs:")
for name, cid in selected:
    print(f"- {name}: id={cid}")

Selected competitions and their IDs:
- USL Championship: id=191


Next I will view which seasons I have for each competition, to narrow down my range. 

In [36]:
# For each selected competition, list seasons available via competition editions
for name, cid in selected:
    editions = client.get_competition_competition_editions(competition_id=cid)

    seasons = sorted({
        ed.get("season_name")
        or (ed.get("season") or {}).get("name")
        or str((ed.get("season") or {}).get("id"))
        or str(ed.get("season_id"))
        for ed in editions
        if ed.get("season_name")
        or (ed.get("season") or {}).get("name")
        or ed.get("season_id")
    })

    print(f"\n{name} (competition_id={cid})")
    print("Seasons you have access to:")
    for s in seasons:
        print(f"  - {s}")


USL Championship (competition_id=191)
Seasons you have access to:
  - 2021
  - 2022
  - 2023
  - 2024
  - 2025
  - 2026


Define season ranges:

> *2021-2025*

View competition edition ID (League + season Combo)

In [37]:
editions = client.get_competition_editions(competition_id=191)
editions_df = pd.DataFrame(editions)

# This DataFrame will show you the 'id' (this is the edition ID), 'name', and 'season'
# Look for the row where name is "USL Championship 2025" or similar.
print(editions_df[['id', 'name', 'season']])

Unexpected keyword argument: competition_id


        id                                               name  \
0        0                            FIFA - World Cup - 2014   
1        1                   ENG - Premier League - 2015/2016   
2        2                UEFA - European Championship - 2016   
3        3                   ENG - Premier League - 2016/2017   
4        4                   ENG - Premier League - 2017/2018   
...    ...                                                ...   
1198  1427                      AFC - CAFA Nations Cup - 2025   
1199  1428  SCO - Scottish Women's Premier League - 2025/2026   
1200  1429                           TUR - 2. Lig - 2025/2026   
1201  1430                       AUT - Frauenliga - 2025/2026   
1202  1431                          CIV - Ligue 1 - 2025/2026   

                                                 season  
0     {'id': 10, 'start_year': 2014, 'end_year': 201...  
1     {'id': 2, 'start_year': 2015, 'end_year': 2016...  
2     {'id': 12, 'start_year': 2016, 'end_yea

In [39]:
editions_df[editions_df['name'].str.contains('USL')]

,id,competition,season,name
445,481,"{'id': 191, 'area': 'USA', 'name': 'USL Champi...","{'id': 20, 'start_year': 2022, 'end_year': 202...",USA - USL Championship - 2022
446,482,"{'id': 191, 'area': 'USA', 'name': 'USL Champi...","{'id': 17, 'start_year': 2021, 'end_year': 202...",USA - USL Championship - 2021
459,495,"{'id': 191, 'area': 'USA', 'name': 'USL Champi...","{'id': 22, 'start_year': 2023, 'end_year': 202...",USA - USL Championship - 2023
671,802,"{'id': 191, 'area': 'USA', 'name': 'USL Champi...","{'id': 29, 'start_year': 2024, 'end_year': 202...",USA - USL Championship - 2024
837,1019,"{'id': 454, 'area': 'USA', 'name': 'USL League...","{'id': 29, 'start_year': 2024, 'end_year': 202...",USA - USL League 1 - 2024
900,1094,"{'id': 191, 'area': 'USA', 'name': 'USL Champi...","{'id': 128, 'start_year': 2025, 'end_year': 20...",USA - USL Championship - 2025
918,1114,"{'id': 489, 'area': 'USA', 'name': 'USL Super ...","{'id': 95, 'start_year': 2024, 'end_year': 202...","USA - USL Super League, Women - 2024/2025"
955,1152,"{'id': 454, 'area': 'USA', 'name': 'USL League...","{'id': 128, 'start_year': 2025, 'end_year': 20...",USA - USL League 1 - 2025
1087,1308,"{'id': 489, 'area': 'USA', 'name': 'USL Super ...","{'id': 129, 'start_year': 2025, 'end_year': 20...","USA - USL Super League, Women - 2025/2026"
1144,1369,"{'id': 191, 'area': 'USA', 'name': 'USL Champi...","{'id': 130, 'start_year': 2026, 'end_year': 20...",USA - USL Championship - 2026


In [44]:
editions_df[editions_df['id'] == 1094]

,id,competition,season,name
900,1094,"{'id': 191, 'area': 'USA', 'name': 'USL Champi...","{'id': 128, 'start_year': 2025, 'end_year': 20...",USA - USL Championship - 2025


> EDITION ID: 1094

# Ingest Data from Selected Season 2025

After further ideation, analysis will only use one season, 2025, as this is more than enough data for the analysis. Any more would only provide more technical overhead and introduce unecessary computing costs. 

## Download From Drive for Desktop

In [8]:
with open("./creds/gdrive_folder.json", 'r') as f:
    gdrive_data_folder_path = json.load(f)["data_folder_path"]

In [63]:
with open(gdrive_data_folder_path + 'skillcorner/tracking/tracking_usl_championship-2025-2006551.json', 'r') as f:
    data = json.load(f)

print(f"Type: {type(data)}")
print(f"Total frames: {len(data)}")
print(f"\nFirst frame:")
print(json.dumps(data[300], indent=2))

# Or preview first 3 frames
print(f"\nFirst 3 frames (summary):")
for i in [100,200,300]:
    frame = data[i]
    print(f"\nFrame {i}: frame={frame.get('frame')}, period={frame.get('period')}, players={len(frame.get('player_data', []))}")

Type: <class 'list'>
Total frames: 63031

First frame:
{
  "frame": 300,
  "timestamp": "00:00:25.00",
  "period": 1,
  "ball_data": {
    "x": -28.4,
    "y": 29.31,
    "z": 2.04,
    "is_detected": true
  },
  "possession": {
    "player_id": null,
    "group": "away team"
  },
  "image_corners_projection": {
    "x_top_left": -47.43,
    "y_top_left": 39.0,
    "x_bottom_left": -20.81,
    "y_bottom_left": -19.11,
    "x_bottom_right": -6.55,
    "y_bottom_right": -14.98,
    "x_top_right": -15.66,
    "y_top_right": 39.0
  },
  "player_data": [
    {
      "x": -47.85,
      "y": 5.32,
      "player_id": 333616,
      "is_detected": false
    },
    {
      "x": -36.16,
      "y": 3.78,
      "player_id": 333032,
      "is_detected": false
    },
    {
      "x": -39.65,
      "y": 19.27,
      "player_id": 333619,
      "is_detected": false
    },
    {
      "x": -30.07,
      "y": -13.0,
      "player_id": 333625,
      "is_detected": false
    },
    {
      "x": -26.42,
     

In [64]:
with open(gdrive_data_folder_path + 'skillcorner/tracking/tracking_usl_championship-2025-2006551.json', 'r') as f:
    data = json.load(f)

# Top-level keys in first frame
first_frame = data[0]
print("Top-level keys:")
print(list(first_frame.keys()))

Top-level keys:
['frame', 'timestamp', 'period', 'ball_data', 'possession', 'image_corners_projection', 'player_data']


In [ ]:
def get_all_keys(obj, prefix=""):
    """Recursively get all keys from nested dictionaries"""
    keys = set()
    if isinstance(obj, dict):
        for key, value in obj.items():
            keys.add(prefix + key if prefix else key)
            if isinstance(value, dict):
                keys.update(get_all_keys(value, prefix + key + "."))
            elif isinstance(value, list) and len(value) > 0 and isinstance(value[0], dict):
                keys.update(get_all_keys(value[0], prefix + key + "[]."))
    return keys

with open(gdrive_data_folder_path + 'skillcorner/tracking/tracking_usl_championship-2025-2006551.json', 'r') as f:
    data = json.load(f)

all_keys = get_all_keys(data[0])
print("All keys (including nested):")
for key in sorted(all_keys):
    print(f"  - {key}")

All keys (including nested):
  - ball_data
  - ball_data.is_detected
  - ball_data.x
  - ball_data.y
  - ball_data.z
  - frame
  - image_corners_projection
  - image_corners_projection.x_bottom_left
  - image_corners_projection.x_bottom_right
  - image_corners_projection.x_top_left
  - image_corners_projection.x_top_right
  - image_corners_projection.y_bottom_left
  - image_corners_projection.y_bottom_right
  - image_corners_projection.y_top_left
  - image_corners_projection.y_top_right
  - period
  - player_data
  - possession
  - possession.group
  - possession.player_id
  - timestamp


In [ ]:
with open(gdrive_data_folder_path + 'skillcorner/tracking/tracking_usl_championship-2025-2006551.json', 'r') as f:
    data = json.load(f)

first_frame = data[3000]

print("Top-level keys:")
print(list(first_frame.keys()))

print("\nNested keys in 'ball_data':")
if 'ball_data' in first_frame:
    print(list(first_frame['ball_data'].keys()))

print("\nNested keys in 'possession':")
if 'possession' in first_frame:
    print(list(first_frame['possession'].keys()))

print("\nKeys in first player (if player_data exists):")
if 'player_data' in first_frame and len(first_frame['player_data']) > 0:
    print(list(first_frame['player_data'][0].keys()))

Top-level keys:
['frame', 'timestamp', 'period', 'ball_data', 'possession', 'image_corners_projection', 'player_data']

Nested keys in 'ball_data':
['x', 'y', 'z', 'is_detected']

Nested keys in 'possession':
['player_id', 'group']

Keys in first player (if player_data exists):
['x', 'y', 'player_id', 'is_detected']


In [ ]:
with open(gdrive_data_folder_path + 'skillcorner/tracking/tracking_usl_championship-2025-2006551.json', 'r') as f:
    data = json.load(f)

first_frame = data[3000]

print("Available keys to access:")
for key in first_frame.keys():
    value = first_frame[key]
    if isinstance(value, dict):
        print(f"  {key} (dict) -> {list(value.keys())}")
    elif isinstance(value, list):
        print(f"  {key} (list, length={len(value)})")
        if len(value) > 0 and isinstance(value[0], dict):
            print(f"    -> First item keys: {list(value[0].keys())}")
    else:
        print(f"  {key} -> {type(value).__name__}")

Available keys to access:
  frame -> int
  timestamp -> str
  period -> int
  ball_data (dict) -> ['x', 'y', 'z', 'is_detected']
  possession (dict) -> ['player_id', 'group']
  image_corners_projection (dict) -> ['x_top_left', 'y_top_left', 'x_bottom_left', 'y_bottom_left', 'x_bottom_right', 'y_bottom_right', 'x_top_right', 'y_top_right']
  player_data (list, length=22)
    -> First item keys: ['x', 'y', 'player_id', 'is_detected']


## Lets have some fun real quick

I will make am moving picture of a few seconds of gametime, to show all the players moving and the ball like I see on LinkedIn. 

In [ ]:
path = gdrive_data_folder_path + 'skillcorner/tracking/tracking_usl_championship-2025-2006551.json'

with open(path, "r") as f:
    frames = json.load(f)

start_frame = 10_000
end_frame   = 10_500
frames_slice = frames[start_frame:end_frame + 1]

print(f"Animating {len(frames_slice)} frames")

# --- Set plot limits ---
x_min, x_max = -60.1, 60.3
y_min, y_max = -41.7, 39.2

# --- Matplotlib figure ---
fig, ax = plt.subplots(figsize=(14, 10))

ax.set_xlim(x_min, x_max)
ax.set_ylim(y_min, y_max)
ax.set_aspect("equal")
ax.set_facecolor("#3f995b")
ax.set_title("USL Championship Tracking – Frames 10,000–11,000", fontsize=16, color="white", weight="bold")

# Draw pitch outline
pitch_length = 105
pitch_width = 68
ax.plot(
    [-pitch_length/2, pitch_length/2, pitch_length/2, -pitch_length/2, -pitch_length/2],
    [-pitch_width/2, -pitch_width/2, pitch_width/2, pitch_width/2, -pitch_width/2],
    color="white",
    linewidth=2.5,
)

ax.axvline(0, color="white", linewidth=1.5, linestyle="--", alpha=0.7)

center_circle = plt.Circle((0, 0), 9.15, fill=False, color="white", linewidth=1.5, linestyle="--", alpha=0.7)
ax.add_patch(center_circle)

# Initialize empty scatter plots
away_scatter = ax.scatter([], [], c="red", s=80, label="Away", edgecolors="white", linewidths=1, zorder=5, alpha=0.9)
home_scatter = ax.scatter([], [], c="blue", s=80, label="Home", edgecolors="white", linewidths=1, zorder=5, alpha=0.9)
ball_scatter = ax.scatter([], [], c="black", s=100, label="Ball", edgecolors="white", linewidths=1.5, zorder=10, alpha=0.95)

timestamp_text = ax.text(
    0.02, 0.97, "", transform=ax.transAxes, 
    color="white", fontsize=13, weight="bold",
    ha="left", va="top",
    bbox=dict(boxstyle="round,pad=0.8", facecolor="black", alpha=0.8, edgecolor="white", linewidth=1.5)
)

ax.legend(loc="upper right", facecolor="white", framealpha=0.95, fontsize=11, edgecolor="black", frameon=True)

# --- Helper function ---
def split_players_by_x(player_data):
    away_x, away_y = [], []
    home_x, home_y = [], []
    for p in player_data:
        if not p.get("is_detected", True):
            continue
        x = p.get("x")
        y = p.get("y")
        if x is None or y is None:
            continue
        if x < 0:
            away_x.append(x)
            away_y.append(y)
        else:
            home_x.append(x)
            home_y.append(y)
    return (away_x, away_y), (home_x, home_y)

# --- Animation update function ---
def update(i):
    frame = frames_slice[i]
    player_data = frame.get("player_data", [])
    (away_x, away_y), (home_x, home_y) = split_players_by_x(player_data)

    if away_x:
        away_offsets = np.column_stack([away_x, away_y])
    else:
        away_offsets = np.empty((0, 2))
    away_scatter.set_offsets(away_offsets)

    if home_x:
        home_offsets = np.column_stack([home_x, home_y])
    else:
        home_offsets = np.empty((0, 2))
    home_scatter.set_offsets(home_offsets)

    ball = frame.get("ball_data", {})
    if ball.get("is_detected") and ball.get("x") is not None and ball.get("y") is not None:
        ball_offsets = np.array([[ball["x"], ball["y"]]], dtype=float)
        ball_scatter.set_offsets(ball_offsets)
        ball_scatter.set_visible(True)
    else:
        ball_scatter.set_offsets(np.empty((0, 2)))
        ball_scatter.set_visible(False)

    ts = frame.get("timestamp", "N/A")
    period = frame.get("period", "N/A")
    frame_num = start_frame + i
    timestamp_text.set_text(
        f"Frame: {frame_num} | Period: {period} | Time: {ts}\n"
        f"Away: {len(away_x)} players | Home: {len(home_x)} players"
    )

# Create animation
anim = FuncAnimation(
    fig,
    update,
    frames=len(frames_slice),
    interval=20,
    repeat=True,
    blit=False,
)

plt.tight_layout()

# Save as MP4 video (requires ffmpeg)
print("Saving animation as video...")
anim.save('tracking_animation_10000_11000.mp4', writer='ffmpeg', fps=20, bitrate=1800)
print("Animation saved as 'tracking_animation_10000_11000.mp4'")

# Also try to display in notebook
plt.show()

Animating 501 frames


<IPython.core.display.Javascript object>

Saving animation as video...
Animation saved as 'tracking_animation_10000_11000.mp4'


# Get Match Metadata

Get and clean matches metadata

Good thing we did! Because we noticed something very important. I need more metadata. Lets get that now.

In [45]:
matches = client.get_matches(params={'competition_edition': 1094})
matches_df = pd.DataFrame(matches)

In [50]:
matches_df

,id,date_time,home_team,away_team,status,competition_id,season_id,competition_edition_id
0,2044556,2025-11-22T17:00:00Z,"{'id': 2715, 'short_name': 'Tulsa'}","{'id': 2748, 'short_name': 'Pittsburgh Rive'}",closed,191,128,1094
1,2043902,2025-11-16T01:30:00Z,"{'id': 2715, 'short_name': 'Tulsa'}","{'id': 2751, 'short_name': 'New Mexico Utd'}",closed,191,128,1094
2,2043833,2025-11-15T23:30:00Z,"{'id': 2748, 'short_name': 'Pittsburgh Rive'}","{'id': 3483, 'short_name': 'Rhode Island'}",closed,191,128,1094
3,2040895,2025-11-09T02:00:00Z,"{'id': 2751, 'short_name': 'New Mexico Utd'}","{'id': 2759, 'short_name': 'Orange County'}",closed,191,128,1094
4,2040896,2025-11-09T01:00:00Z,"{'id': 2715, 'short_name': 'Tulsa'}","{'id': 2766, 'short_name': 'Phoenix Rising'}",closed,191,128,1094
...,...,...,...,...,...,...,...,...
369,2006779,2025-03-09T01:00:00Z,"{'id': 2767, 'short_name': 'Birmingham L'}","{'id': 2749, 'short_name': 'Loudoun United'}",closed,191,128,1094
370,2006780,2025-03-09T00:30:00Z,"{'id': 2753, 'short_name': 'Charleston Bat'}","{'id': 2756, 'short_name': 'Louisville City'}",closed,191,128,1094
371,2006557,2025-03-09T00:00:00Z,"{'id': 4178, 'short_name': 'Lexington SC'}","{'id': 2755, 'short_name': 'Hartford'}",closed,191,128,1094
372,2006558,2025-03-09T00:00:00Z,"{'id': 2758, 'short_name': 'Miami'}","{'id': 2746, 'short_name': 'Detroit City'}",closed,191,128,1094


In [52]:
matches_df['id'].nunique()

374

In [65]:
len(os.listdir(gdrive_data_folder_path + "skillcorner/tracking/"))

358

I need to filter the matches_df to keep only the games I actually was able to download. 

In [ ]:
# Folder where your tracking JSON files live
tracking_dir = Path(gdrive_data_folder_path + "skillcorner/tracking/")

# List filenames, extract match ID (last number before .json)
file_names = [f.name for f in tracking_dir.glob("*.json")]
match_ids_from_files = [int(f.replace(".json", "").split("-")[-1]) for f in file_names]

print(f"Found {len(match_ids_from_files)} match IDs in folder")
print("Sample:", match_ids_from_files[:5])

# Filter matches_df to only rows whose match ID is in the downloaded set
matches_df = matches_df[matches_df["id"].isin(match_ids_from_files)].reset_index(drop=True)

print(f"matches_df now has {len(matches_df)} rows (only matches with tracking data)")

Found 358 match IDs in folder
Sample: [2012349, 2015033, 2022439, 2008482, 2016473]
matches_df now has 358 rows (only matches with tracking data)


In [59]:
matches_df.shape

(358, 8)

In [ ]:
matches_df.to_csv(gdrive_data_folder_path + "skillcorner/matches_df.csv", index=False)

Next step, make sure all match IDs match.

In [67]:
set(matches_df["id"]) == set(match_ids_from_files)

True

# Player Metadata

Perfect! All good to go. Next step, we need to get the player metadata. I will do that with the `get_match()` function. 

In [9]:
matches_df = pd.read_csv(gdrive_data_folder_path + "skillcorner/matches_df.csv")

In [10]:
matches_df.columns

Index(['id', 'date_time', 'home_team', 'away_team', 'status', 'competition_id',
       'season_id', 'competition_edition_id'],
      dtype='str')

In [73]:
specific_match_ids = matches_df['id'].head(3).tolist()
print(f"Processing matches: {specific_match_ids}") 

player_mapping_list = []

for match_id in specific_match_ids:
    print(f"Fetching match: {match_id}")
    try:
        match_metadata = client.get_match(match_id)
        
        # Helper function to extract players safely
        def extract_players(metadata, side):
            # Try 'home_team_kit' or 'away_team_kit' first
            kit_key = f"{side}_team_kit"
            team_key = f"{side}_team"
            
            # Get Team ID safely
            team_id = metadata.get(team_key, {}).get('id', None)
            
            players_found = []
            
            # Try to find the player list
            # Path A: metadata['home_team_kit']['players']
            if kit_key in metadata and 'players' in metadata[kit_key]:
                players_found = metadata[kit_key]['players']
            
            # Path B: metadata['home_team']['players'] (Alternative structure)
            elif team_key in metadata and 'players' in metadata[team_key]:
                players_found = metadata[team_key]['players']

            # Process valid players
            for player in players_found:
                player_mapping_list.append({
                    'match_id': match_id,
                    'player_id': player.get('player_id') or player.get('id'),
                    'player_name': f"{player.get('first_name','')} {player.get('last_name','')}",
                    'team_id': team_id,
                    'team_side': side
                })
        
        extract_players(match_metadata, 'home')
        extract_players(match_metadata, 'away')

    except Exception as e:
        print(f"Error on match {match_id}: {e}")

players_df = pd.DataFrame(player_mapping_list)
print(f"Found {len(players_df)} player records.")
print(players_df.head())

Processing matches: [2044556, 2043902, 2043833]
Fetching match: 2044556
Fetching match: 2043902
Fetching match: 2043833
Found 0 player records.
Empty DataFrame
Columns: []
Index: []


In [77]:
# Grab the first ID from your list
test_id = 2044556 
print(f"Inspecting match: {test_id}")

try:
    data = client.get_match(test_id)
    print("Top level keys:", data.keys())
    
    if 'home_team_kit' in data:
        print("Home Kit keys:", data['home_team_kit'].keys())
    if 'home_team' in data:
        print("Home Team keys:", data['home_team'].keys())
    if 'players' in data:
        print("Players keys:", data['players'])
        
except Exception as e:
    print(f"Failed to fetch: {e}")

Inspecting match: 2044556
Top level keys: dict_keys(['id', 'home_team_score', 'away_team_score', 'date_time', 'stadium', 'home_team', 'home_team_kit', 'away_team', 'away_team_kit', 'home_team_coach', 'away_team_coach', 'home_team_playing_time', 'away_team_playing_time', 'competition_edition', 'match_periods', 'competition_round', 'referees', 'players', 'status', 'home_team_side', 'ball', 'pitch_length', 'pitch_width'])
Home Kit keys: dict_keys(['id', 'team_id', 'season', 'name', 'jersey_color', 'number_color'])
Home Team keys: dict_keys(['id', 'name', 'short_name', 'acronym'])
Players keys: [{'player_role': {'id': 6, 'position_group': 'Full Back', 'name': 'Right Wing Back', 'acronym': 'RWB'}, 'start_time': '01:30:02', 'end_time': None, 'number': 47, 'yellow_card': 0, 'red_card': 0, 'injured': False, 'goal': 0, 'own_goal': 0, 'playing_time': {'total': {'minutes_tip': 8.94, 'minutes_otip': 6.08, 'start_frame': 57300, 'end_frame': 76039, 'minutes_played': 31.15, 'minutes_played_regular_ti

In [ ]:
player_mapping_list = []

print(f"Processing {len(matches_df)} matches...")

for match_id in matches_df['id']:
    try:
        # 1. Fetch the match data
        match_metadata = client.get_match(match_id)
        
        # 2. Check if the top-level 'players' key exists
        if 'players' in match_metadata:
            # 3. Iterate through the single list containing BOTH teams' players
            for player in match_metadata['players']:
                player_mapping_list.append({
                    'match_id': match_id,
                    'player_id': player['id'],  # Note: It's 'id' here, not 'player_id'
                    'player_name': f"{player.get('first_name', '')} {player.get('last_name', '')}",
                    'team_id': player['team_id'], # This is the magic link you needed
                    'number': player.get('number'),
                    'position': player.get('player_role', {}).get('name')
                })
        else:
            print(f"Warning: No 'players' list found for match {match_id}")

    except Exception as e:
        print(f"Error processing match {match_id}: {e}")

# 4. Create the DataFrame
players_df = pd.DataFrame(player_mapping_list)

print(f"Success! Found {len(players_df)} player records.")
print(players_df.head())

# 5. Save to the designated Google Drive folder
# Define the target directory
target_directory = os.path.join(gdrive_data_folder_path, 'skillcorner')

# Ensure the directory exists (creates it if it doesn't)
os.makedirs(target_directory, exist_ok=True)

# Define the full file path including the file name
file_path = os.path.join(target_directory, 'usl_player_match_map.csv')

# Save the DataFrame to CSV
players_df.to_csv(file_path, index=False)

print(f"Data successfully saved to: {file_path}")

Processing 358 matches...
